# Seed 42

# Eye Movement-Based Schizophrenia Recognition — Full Pipeline

| Cell | Mục đích |
|---|---|
| 1 | Mount Drive + cd vào project |
| 2 | 🔴 XÓA kết quả cũ (bỏ comment khi cần reset) |
| 3 | Install thư viện còn thiếu |
| 4 | Tier 1 — Preprocessing |
| 5 | Tier 2 — Feature Engineering |
| 6 | Tier 3 — Tabular (XGBoost) |
| 7 | Tier 4A — ResNet50 extraction |
| 8 | Tier 4B — Build graphs |
| 9 | Tier 4C — GNN-CEFAM training |
| 10 | Tier 4D — BiCA-HS training |
| 11 | Tier 5 — Meta-Learner |
| 12 | Tổng hợp kết quả |

# Drive + lib

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition


In [ ]:
# 🔴 XÓA KẾT QUẢ CŨ — bỏ comment từng dòng tùy mức độ reset

# Xóa checkpoint + OOF Tier 4 (BẮT BUỘC sau khi fix C-1, C-2, C-3)
# !rm -rf results/bica/ results/cefam/ results/stgnn/ results/tier5/
# !rm -rf "Bidirectional Cross-Attention Hybrid Stream/results/checkpoints/"

# Xóa Tier 3
# !rm -rf results/baselines/

# Xóa graphs (rebuild từ đầu)
# !rm -rf data/processed/graphs/

# Xóa toàn bộ (chạy lại từ raw data)
# !rm -rf results/ data/processed/ data/external/

In [2]:
# Colab đã có sẵn torch/sklearn/pandas — chỉ install thêm cái còn thiếu
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm shap

# Tier 1: Once for all seed

In [ ]:
# Tier 1: Tạo category map + tiền xử lý dữ liệu thô
!python -m src.utils.generate_category_map
!python -m src.tier1_preprocessing.preprocess

Successfully generated category mapping for 100 images at data/metadata/stimulus_categories.csv
--- Tier 1: Loading raw data ---
Loading Fixations: 100% 160/160 [00:18<00:00,  8.66it/s]
Loading Fixations: 100% 48/48 [00:36<00:00,  1.31it/s]
Total raw fixations loaded: 293740
Spatial boundary filter: removed 5037 out of 293740 fixations (1.71%)
Temporal duration filter: removed 7666 out of 288703 fixations (2.66%)
Successfully saved 281037 fixations to data/processed/clean_fixations.parquet
--- Tier 1 Preprocessing Completed Successfully ---


# Tier 2: Once for all seed

In [3]:
# Tier 2: Trích xuất đặc trưng stimulus-level + subject-level delta
!python -m src.tier2_features.stimulus_features
!python -m src.tier2_features.subject_aggregator

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Extracting features per trial...
100% 20695/20695 [00:23<00:00, 871.68it/s]
Extracted features for 20695 trials. Saved to data/processed/features_stimulus_level.csv
Loading stimulus-level features from data/processed/features_stimulus_level.csv...
Loading category mapping from data/metadata/stimulus_categories.csv...
Computing mean feature values per subject, per category...
Computing contextual delta features...
Aggregated subject-level features for 208 subjects. Saved to data/processed/features_subject_level.csv


# Seed 42

# Tier 3 - Seed 42: XGBoost, LightGBM, CatBoost

In [3]:
# Tier 3: Tabular baseline (XGBoost)
!python -m src.tier3_tabular.tabular_models --model xgboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:06:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_e

In [4]:
!python -m src.tier3_tabular.tabular_models --model lightgbm

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7615
  Stimulus-Level Auc: 0.8385
  Stimulus-Level 

In [5]:
!python -m src.tier3_tabular.tabular_models --model catboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.8114
  Stimulus-Level Auc: 0.8971
  Stimulus-Level 

# Tier 4A: Extract ResNet feature, once for all seed

In [ ]:
# Tier 4A: Trích xuất ResNet50 visual features (2048-dim)
# Output: data/external/feature_dict_ResNet50.npy
!python -m src.utils.extract_resnet_features

 VISUAL FEATURE EXTRACTION (ResNet50 Baseline)
Found 100 stimulus images in EMS/Images.
Loading pre-trained ResNet50 on device: cuda...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 242MB/s]
Extracting feature maps for all stimulus images...
Extracting image features: 100% 100/100 [01:12<00:00,  1.38it/s]
Mapping visual features to subject fixations...
Mapping to trials: 100% 16716/16716 [00:56<00:00, 295.45it/s]

Successfully extracted ResNet50 features. Saved to data/external/feature_dict_ResNet50.npy
Total trials mapped: 16716
Feature vector dimension: 2048


# Tier 4B: Build graph, once for all seed

In [ ]:
# Tier 4B: Xây dựng đồ thị PyG từ fixation data + ResNet50 features
# Output: data/processed/graphs/graphs.pt
!python -m src.tier4_advanced.graph_builder

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Global pupil stats (minor leakage — see NOTE above): Mean=1218.29, Std=614.17
Loading ResNet50 features from data/external/feature_dict_ResNet50.npy...
Detected visual feature dimension from dataset: 2048
Building spatiotemporal graphs...
100% 16716/16716 [00:37<00:00, 448.59it/s]
Successfully constructed and saved 16716 graphs at data/processed/graphs/graphs.pt


# All tier 4 - Seed 42

In [3]:
# Tier 4C: Huấn luyện GNN-CEFAM (4-fold GroupKFold)
# Output: results/cefam/cefam_oof_subject_preds.csv
!python scripts/train_tier4.py --seed 42 --batch-size 128

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Found existing checkpoint at results/cefam/checkpoints/cefam_fold_0_best.pt. Evaluating...
Loaded checkpoin

In [8]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml --seed 42 --batch-size 128

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0751 TrAUC=0.7604 | VaLoss=0.0716 VaTrialAUC=0.7916 VaSubjAUC=0.8485
Ep 005/200 | TrLoss=0.0431 TrAUC=0.9043 | VaLoss=0.0599 VaTrialAUC=0.8171 VaSubjAUC=0.8712
Ep 010/200 | TrLoss=0.0392 TrAUC=0.9207 | VaLoss=0.0714 VaTrialAUC=0.8290 VaSubjAUC=0.8864
Ep 015/200 | TrLoss=0.0338 TrAUC=0.9407 | VaLoss=0.0839 VaTrialAUC

In [ ]:
# Old version wrong 15 feature, ran new version with 135 feature below
# Tier 4D: Huấn luyện BiCA-HS (4-fold GroupKFold)
# PYTHONPATH=. bắt buộc để import src.*
# Output: results/bica/bica_subject_val_predictions.csv
# !PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --config configs/bica_config.yaml

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Found existing checkpoint at results/bica/checkpoints/bica_fold_0_best.pt. Resuming and evaluating...
Loaded checkpoint - Val Loss: 0.0595 | Val Trial AUC: 0.8669 | Val Subject AUC: 0.9141

==================== Training Fold 1 ====================
Train trials: 11708, Val trials: 3890
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/s

In [2]:
!python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" \
    --seed 42 \
    --batch-size 128 \
    --output_dir results/bica_135feat_s42

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train trials: 11686, Val trials: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0433 | Train AUC: 0.9569 | Val Loss: 0.0963 | Val Trial AUC: 0.8360 | Val

# All tier 5 - seed 42

In [3]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s42/xgboost_oof_subject_preds.csv \
    --tier3-test results/baselines_s42/xgboost_test_predictions.csv \
    --tier4-oof results/cefam_s42/cefam_oof_subject_preds.csv \
    --tier4-test results/cefam_s42/cefam_test_predictions.csv \
    --output-dir results/tier5_cefam_s42/ \
    --plot --calibrate

2026-06-29 10:47:56.469714: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:47:56.535125: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.001, 0.996]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9119  ACC:

In [4]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s42/xgboost_oof_subject_preds.csv \
    --tier3-test results/baselines_s42/xgboost_test_predictions.csv \
    --tier4-oof results/bica_s42/bica_subject_val_predictions.csv \
    --tier4-test results/bica_s42/bica_test_predictions.csv \
    --output-dir results/tier5_bica_s42/ \
    --plot --calibrate

2026-06-29 10:48:33.285058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:48:33.355596: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.001, 0.999]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9581  ACC:

# Seed 123

# Tier 3 - Seed 123: XGBoost, LightGBM, CatBoost

In [5]:
!python -m src.tier3_tabular.tabular_models --model xgboost --seed 123
!python -m src.tier3_tabular.tabular_models --model catboost --seed 123

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:00:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_e

In [6]:
!python -m src.tier3_tabular.tabular_models --model lightgbm --seed 123

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7866
  Stimulus-Level Auc: 0.8430
  Stimulus-Level 

# All tier 4 - seed 123

In [4]:
!python scripts/train_tier4.py --seed 123 --batch-size 128

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0537 TrAUC=0.9359 | VaLoss=0.0853 VaTrialAUC=0.8608 VaSubjAUC=0.8763
Ep 005/200 | TrL

In [8]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml --seed 123 --batch-size 128

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0760 TrAUC=0.7676 | VaLoss=0.0767 VaTrialAUC=0.7602 VaSubjAUC=0.8030
Ep 005/200 | TrLoss=0.0448 TrAUC=0.8974 | VaLoss=0.0704 VaTrialAUC=0.8037 VaSubjAUC=0.8409
Ep 010/200 | TrLoss=0.0402 TrAUC=0.9157 | VaLoss=0.0641 VaTrialAUC=0.8084 VaSubjAUC=0.8535
Ep 015/200 | TrLoss=0.0378 TrAUC=0.9265 | VaLoss=0.0728 VaTrialAUC

In [ ]:
# Old with 15 feature, new version with 135 feature ran below
#!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" \
#    --config configs/bica_config.yaml --seed 123

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0624 | Train AUC: 0.8908 | Val Loss: 0.0922 | Val Trial AUC: 0.7297 | Val Subject AUC: 0.7374
Epoch 005/150 | Train Loss: 0.0436 | Train AUC: 0.9268 | Val Loss: 0.0677 | Val Trial AUC: 0.8406 | Val Subject AUC: 0.8864
Epoch 010/150 | Train Loss: 0.0423 | Train AUC: 0.9315 | Val Loss: 0.0872 | Val Trial AUC: 0.8223 | Val Sub

In [3]:
!python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --seed 123 --output_dir results/bica_135feat_s123

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train trials: 11686, Val trials: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0424 | Train AUC: 0.9535 | Val Loss: 0.0489 | Val Trial AUC: 0.9165 | Val

# All tier 5 - seed 123

In [12]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s123/xgboost_oof_subject_preds.csv \
    --tier3-test results/baselines_s123/xgboost_test_predictions.csv \
    --tier4-oof results/cefam_s123/cefam_oof_subject_preds.csv \
    --tier4-test results/cefam_s123/cefam_test_predictions.csv \
    --output-dir results/tier5_cefam_s123/ \
    --plot --calibrate

2026-06-29 10:57:07.656295: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:57:07.723127: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.002, 0.999]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8667  ACC: 0.7625  Brier: 0.1825  BSS: 0.2699  ECE: 0.1805

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9264  ACC:

In [16]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s123/xgboost_oof_subject_preds.csv \
    --tier3-test results/baselines_s123/xgboost_test_predictions.csv \
    --tier4-oof results/bica_s123/bica_subject_val_predictions.csv \
    --tier4-test results/bica_s123/bica_test_predictions.csv \
    --output-dir results/tier5_bica_s123/ \
    --plot --calibrate

2026-06-29 10:58:34.298177: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:58:34.364421: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.000, 0.995]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8667  ACC: 0.7625  Brier: 0.1825  BSS: 0.2699  ECE: 0.1805

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9572  ACC:

# Seed 456

# Tier 3 - Seed 456: XGBoost, LightGBM, CatBoost

In [11]:
!python -m src.tier3_tabular.tabular_models --model xgboost --seed 456
!python -m src.tier3_tabular.tabular_models --model catboost --seed 456

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:42:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_e

In [12]:
!python -m src.tier3_tabular.tabular_models --model lightgbm --seed 456

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7803
  Stimulus-Level Auc: 0.8453
  Stimulus-Level 

# All tier 4 - seed 456

In [5]:
!python scripts/train_tier4.py --seed 456 --batch-size 128

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1579, in load
    return _load(
           ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 2190, in _load
    result = unpickler.load()
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 2154, in persistent_load
    typed_storage = load_tensor(
                    ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 2097, in load_tensor
    zip_file.get_storage_from_record(name, nbytes, torch.UntypedStorage)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TM

In [14]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml --seed 456 --batch-size 128

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0739 TrAUC=0.7865 | VaLoss=0.0740 VaTrialAUC=0.7802 VaSubjAUC=0.8384
Ep 005/200 | TrLoss=0.0444 TrAUC=0.8970 | VaLoss=0.0822 VaTrialAUC=0.7840 VaSubjAUC=0.8157
Ep 010/200 | TrLoss=0.0405 TrAUC=0.9158 | VaLoss=0.0833 VaTrialAUC=0.8101 VaSubjAUC=0.8586
Ep 015/200 | TrLoss=0.0361 TrAUC=0.9329 | VaLoss=0.0844 VaTrialAUC

In [ ]:
# Old with 15 feature, new version with 135 feature ran below
#!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" \
#    --config configs/bica_config.yaml --seed 456

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0604 | Train AUC: 0.8969 | Val Loss: 0.0737 | Val Trial AUC: 0.8146 | Val Subject AUC: 0.8460
Epoch 005/150 | Train Loss: 0.0431 | Train AUC: 0.9294 | Val Loss: 0.0647 | Val Trial AUC: 0.8425 | Val Subject AUC: 0.8813
Epoch 010/150 | Train Loss: 0.0409 | Train AUC: 0.9356 | Val Loss: 0.0883 | Val Trial AUC: 0.8389 | Val Sub

In [6]:
!python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --seed 456 --batch-size 128 --output_dir results/bica_135feat_s456

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train trials: 11686, Val trials: 3912
/content/drive/.shortcut-targets-by-id/1fhPJhVsKHZIH9TMR4_HSLuhHEl0bG-Vz/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Found existing checkpoint at results/bica_135feat_s456/checkpoints/bica_fold_0_best.pt. Resuming and ev

# All tier 5 - seed 456

In [14]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s456/xgboost_oof_subject_preds.csv \
    --tier3-test results/baselines_s456/xgboost_test_predictions.csv \
    --tier4-oof results/cefam_s456/cefam_oof_subject_preds.csv \
    --tier4-test results/cefam_s456/cefam_test_predictions.csv \
    --output-dir results/tier5_cefam_s456/ \
    --plot --calibrate

2026-06-29 10:57:47.593727: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:57:47.659711: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.004, 0.993]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8778  ACC: 0.7875  Brier: 0.1670  BSS: 0.3322  ECE: 0.1556

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9241  ACC:

In [17]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines_s456/xgboost_oof_subject_preds.csv \
    --tier3-test results/baselines_s456/xgboost_test_predictions.csv \
    --tier4-oof results/bica_s456/bica_subject_val_predictions.csv \
    --tier4-test results/bica_s456/bica_test_predictions.csv \
    --output-dir results/tier5_bica_s456/ \
    --plot --calibrate

2026-06-29 10:59:01.516816: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 10:59:01.581571: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.000, 0.995]
[Test] Test labels are unknown (Label=-1) — skipping test evaluation.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8778  ACC: 0.7875  Brier: 0.1670  BSS: 0.3322  ECE: 0.1556

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9494  ACC:

# Final analysis

In [9]:
!python scripts/aggregate_multiseed.py

Model                                   AUC              ACC               F1             Sens             Spec
XGBoost                       0.8716±0.0046    0.7771±0.0106    0.7765±0.0120    0.7750±0.0204    0.7792±0.0156
CatBoost                      0.8990±0.0012    0.8104±0.0078    0.8052±0.0076    0.7833±0.0059    0.8375±0.0102
ST-GNN                        0.9331±0.0086    0.7896±0.0029    0.7506±0.0038    0.6333±0.0059    0.9458±0.0059
GNN-CEFAM                     0.9208±0.0064    0.8458±0.0106    0.8465±0.0094    0.8500±0.0102    0.8417±0.0212
BiCA-HS                       0.9549±0.0039    0.8729±0.0128    0.8726±0.0173    0.8750±0.0468    0.8708±0.0212
Tier5 (Ensemble+CEFAM)        0.9387±0.0080    0.8625±0.0088    0.8589±0.0105    0.8375±0.0204    0.8875±0.0102
Tier5 (Ensemble+BiCA-HS)      0.9590±0.0009    0.8854±0.0029    0.8856±0.0043    0.8875±0.0306    0.8833±0.0312
SOTA (MSNet 2025)                    0.8854           0.8125

--- Per-seed AUC ---
  XGBoost            

In [18]:
!python scripts/bootstrap_ci_vs_sota.py

Bootstrap 95% CI (n=10,000) — vs SOTA AUC = 0.8854

XGBoost
    Seed |     AUC |             95% CI |   p-value | Sig?
  -------+---------+--------------------+-----------+------
      42 | 0.8702 | [0.8125, 0.9214] |    0.6934 | NO
     123 | 0.8667 | [0.8081, 0.9187] |    0.7380 | NO
     456 | 0.8778 | [0.8220, 0.9252] |    0.6016 | NO
    POOL |        | [0.8133, 0.9225] |    0.6777 | NO

CatBoost
    Seed |     AUC |             95% CI |   p-value | Sig?
  -------+---------+--------------------+-----------+------
      42 | 0.8981 | [0.8462, 0.9427] |    0.2888 | NO
     123 | 0.9006 | [0.8490, 0.9439] |    0.2604 | NO
     456 | 0.8981 | [0.8469, 0.9423] |    0.2884 | NO
    POOL |        | [0.8470, 0.9430] |    0.2792 | NO

ST-GNN
    Seed |     AUC |             95% CI |   p-value | Sig?
  -------+---------+--------------------+-----------+------
      42 | 0.9378 | [0.8972, 0.9721] |    0.0090 | YES
     123 | 0.9211 | [0.8759, 0.9597] |    0.0585 | NO
     456 | 0.9405 | [0.9

In [19]:
!rm -rf experiments/ablation/results/ experiments/ablation/figures/

!python experiments/ablation/run_ablation_analysis.py

!python experiments/ablation/plot_ablation.py

ABLATION STUDY - Eye Movement-Based Schizophrenia Recognition

Loading model predictions...
Loaded 5 models: ['GNN+CEFAM (Full Hybrid)', 'ST-GNN (GNN Only)', 'BiCA-HS (Transformer)', 'XGBoost (Tabular Only)', 'CatBoost (Tabular Only)']

F1: FULL MODEL COMPARISON (Main Ablation Table)

--- GNN+CEFAM (Full Hybrid) ---
  AUC-ROC:  0.9119
  ACC @0.5: 0.8313
  F1  @0.5: 0.8344
  ACC @opt: 0.8500 (th=0.3517)
  F1  @opt: 0.8571
  Sens@opt: 0.9000
  Spec@opt: 0.8000

--- ST-GNN (GNN Only) ---
  AUC-ROC:  0.9378
  ACC @0.5: 0.7875
  F1  @0.5: 0.7463
  ACC @opt: 0.8750 (th=0.4008)
  F1  @opt: 0.8837
  Sens@opt: 0.9500
  Spec@opt: 0.8000

--- BiCA-HS (Transformer) ---
  AUC-ROC:  0.9581
  ACC @0.5: 0.8750
  F1  @0.5: 0.8765
  ACC @opt: 0.8875 (th=0.5579)
  F1  @opt: 0.8875
  Sens@opt: 0.8875
  Spec@opt: 0.8875

--- XGBoost (Tabular Only) ---
  AUC-ROC:  0.8702
  ACC @0.5: 0.7812
  F1  @0.5: 0.7853
  ACC @opt: 0.8063 (th=0.5953)
  F1  @opt: 0.8050
  Sens@opt: 0.8000
  Spec@opt: 0.8125

--- CatBoos